# NHL Tracking Data Calculations

This notebook uses the available mock tracking CSV to practice simple hockey data analysis. The ideas are intentionally small: clean the tracking data, calculate basic skating metrics, compare teams, and estimate which player is closest to the puck at each timestamp.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.float_format', '{:.4f}'.format)

data = pd.read_csv('trackingData_v2.csv')

# The mock CSV has one unicode minus sign, so normalize it before numeric conversion.
numeric_cols = ['Location.X', 'Location.Y', 'velocity_x', 'velocity_y', 'UTCTime']
for col in numeric_cols:
    data[col] = (
        data[col]
        .astype(str)
        .str.replace('−', '-', regex=False)
        .pipe(pd.to_numeric, errors='coerce')
    )

data['OnPlayingSurface'] = data['OnPlayingSurface'].astype(bool)
running = data[data['ClockState'] == 'ClockStateRunning'].copy()

players = running[running['entityType'] == 'player'].copy()
puck = running[running['entityType'] == 'puck'].copy()

players['elapsed_seconds'] = (players['UTCTime'] - players['UTCTime'].min()) / 1000
puck['elapsed_seconds'] = (puck['UTCTime'] - puck['UTCTime'].min()) / 1000

print(f'Rows in file: {len(data)}')
print(f'Running rows used: {len(running)}')
print(f'Players tracked: {players['player'].nunique()}')
display(players.head())

## Basic Skating Metrics

Distance, speed, and acceleration are calculated between each player's consecutive tracking points. `UTCTime` is stored in milliseconds, so the notebook converts it to seconds before calculating speed.

In [ ]:
players = players.sort_values(['player', 'UTCTime']).copy()

players['dx'] = players.groupby('player')['Location.X'].diff()
players['dy'] = players.groupby('player')['Location.Y'].diff()
players['dt_seconds'] = players.groupby('player')['UTCTime'].diff() / 1000
players['distance'] = np.sqrt(players['dx']**2 + players['dy']**2)
players['speed'] = players['distance'] / players['dt_seconds']
players['acceleration'] = players.groupby('player')['speed'].diff() / players['dt_seconds']

player_summary = (
    players.groupby(['team', 'player'], as_index=False)
    .agg(
        tracking_points=('timeStamp', 'count'),
        total_distance=('distance', 'sum'),
        average_speed=('speed', 'mean'),
        max_speed=('speed', 'max'),
        max_acceleration=('acceleration', 'max')
    )
    .fillna(0)
)

# A simple ranking score: reward max speed first, then acceleration.
player_summary['simple_skater_score'] = (
    player_summary['max_speed'].rank(pct=True) * 0.7
    + player_summary['max_acceleration'].rank(pct=True) * 0.3
)

player_summary = player_summary.sort_values('simple_skater_score', ascending=False)
display(player_summary)

## Team-Level View

This rolls the player metrics up by team. With real tracking data, this could be a starting point for comparing pace, pressure, or workload.

In [ ]:
team_summary = (
    player_summary.groupby('team', as_index=False)
    .agg(
        players=('player', 'count'),
        team_distance=('total_distance', 'sum'),
        average_player_speed=('average_speed', 'mean'),
        fastest_player_speed=('max_speed', 'max')
    )
    .sort_values('team_distance', ascending=False)
)

display(team_summary)

## Simple Puck Proximity / Possession Estimate

A beginner-friendly way to estimate possession from tracking data is to mark the player closest to the puck at each timestamp. This is not the same as official possession, but it is a useful first experiment.

In [ ]:
puck_positions = puck[['timeStamp', 'Location.X', 'Location.Y']].rename(
    columns={'Location.X': 'puck_x', 'Location.Y': 'puck_y'}
)

player_puck = players.merge(puck_positions, on='timeStamp', how='inner')
player_puck['distance_to_puck'] = np.sqrt(
    (player_puck['Location.X'] - player_puck['puck_x'])**2
    + (player_puck['Location.Y'] - player_puck['puck_y'])**2
)

closest_to_puck = (
    player_puck.sort_values(['timeStamp', 'distance_to_puck'])
    .groupby('timeStamp', as_index=False)
    .first()[['timeStamp', 'team', 'player', 'distance_to_puck']]
)

team_possession_estimate = (
    closest_to_puck['team']
    .value_counts(normalize=True)
    .mul(100)
    .rename_axis('team')
    .reset_index(name='estimated_puck_control_pct')
)

display(closest_to_puck)
display(team_possession_estimate)

## Quick Visual Checks

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

player_summary.sort_values('total_distance').plot(
    x='player', y='total_distance', kind='barh', ax=axes[0], legend=False, color='#2f80ed'
)
axes[0].set_title('Total Distance by Player')
axes[0].set_xlabel('Feet')

closest_to_puck.plot(
    x='timeStamp', y='distance_to_puck', marker='o', ax=axes[1], legend=False, color='#27ae60'
)
axes[1].set_title('Closest Player Distance to Puck')
axes[1].set_xlabel('Timestamp')
axes[1].set_ylabel('Feet')

plt.tight_layout()
plt.show()